# Why AI Benchmarks Had to Go End-to-End
Companion notebook for the 2026-09-28 post on MLPerf Inference v6.1.
With pure numpy we reproduce the argument behind the round's biggest change:
**per-stage speedups lie about the pipeline**, so the benchmark now times the
whole RAG pipeline and a full agentic coding session end to end.

*Pipeline latencies below are a labeled toy model (illustrative).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

STAGES = ['embed (e5-base-v2)', 'retrieve (FAISS HNSW)', 'rerank (ColBERTv2)',
          'decompose (gpt-oss-120B)', 'grade docs (gpt-oss-20B)', 'answer (gpt-oss-120B)']
# Toy per-stage latencies in seconds — illustrative, not MLPerf measurements.
T0 = np.array([0.008, 0.025, 0.150, 0.600, 0.400, 2.500])
E2E0 = T0.sum()
def e2e_gain(speedups):
    """End-to-end speedup for per-stage speedup factors (same order as STAGES)."""
    return E2E0 / (T0 / np.asarray(speedups)).sum()
print(f'baseline end-to-end latency: {E2E0:.3f}s')
print('answer stage share: %.1f%%' % (100 * T0[-1] / E2E0))

## 1. Amdahl's law on the RAG pipeline
Speed up *only* the answer-generation stage (a faster GPU, speculative decoding…)
and watch what happens to the end-to-end number. Even an infinitely fast LLM


In [ ]:
s = np.linspace(1, 20, 200)
p = T0[-1] / E2E0
gain = 1.0 / ((1 - p) + p / s)
ceiling = 1.0 / (1 - p)
print(f'3x on the answer stage -> {e2e_gain([1,1,1,1,1,3.0]):.2f}x end-to-end')
print(f'even an INFINITELY fast LLM -> at most {ceiling:.2f}x end-to-end')
plt.plot(s, gain); plt.axhline(ceiling, ls='--')
plt.xlabel('speedup on answer stage only'); plt.ylabel('end-to-end speedup')
plt.title("Amdahl's law: 3x the LLM, <2x the pipeline"); plt.grid(alpha=0.3); plt.show()
# The point of MLPerf's End-to-End RAG test: it reports THIS number, not the stage number.

## 2. The per-stage trap: optimize the right stage
A 3x speedup on one stage vs a balanced set of modest speedups across stages.


In [ ]:
g_single = e2e_gain([1, 1, 1, 1, 1, 3.0])
g_balanced = e2e_gain([2.0, 2.0, 4.0, 2.0, 3.0, 2.0])  # 2x/2x/4x/2x/3x/2x
print(f'3x on answer only        -> {g_single:.2f}x end-to-end')
print(f'balanced across stages   -> {g_balanced:.2f}x end-to-end')
print('Vendor headline says 3x. The user feels %.2f. MLPerf v6.1 reports the user number.' % g_single)

## 3. The agentic edge test: why 'thinking off'
The new benchmark replays 1,007 turns of real SWE-bench Verified coding trajectories
in a single stream and reports mean latency per turn. Toy model: per-turn latency =


In [ ]:
rng = np.random.default_rng(7)
TURNS = 1007  # real number of turns in the v6.1 agentic edge workload
ttft = rng.lognormal(np.log(0.35), 0.5, TURNS)   # time to first token, s
ntok = rng.lognormal(np.log(45), 0.7, TURNS)    # output tokens per turn
TPOT = 0.05  # s per output token (toy)
lat_off = ttft + TPOT * ntok          # thinking off, as the benchmark runs
lat_on = ttft + TPOT * ntok * 5.0     # thinking on: ~5x the output tokens
print(f'thinking OFF: mean {lat_off.mean():.2f}s/turn, replay {lat_off.sum()/60:.1f} min')
print(f'thinking ON : mean {lat_on.mean():.2f}s/turn ({lat_on.mean()/lat_off.mean():.1f}x worse)')
plt.hist(lat_off, bins=60); plt.axvline(lat_off.mean(), ls='--', c='r')
plt.xlabel('seconds per turn'); plt.ylabel('turns')
plt.title('Simulated per-turn latency, 1,007 turns (thinking off)'); plt.show()
# Real anchors from the round: Atlas ran this on an NVIDIA DGX Spark (20.1 tok/s)
# and an AMD Strix Halo desktop (19.63 tok/s) — near parity, all 1,007 turns < 64 min.

## References
- MLCommons chairs blog, 'Where the Industry Is Investing: A Look at MLPerf Inference v6.1' (Sept 16, 2026)
- MLCommons press release, GlobeNewswire (Sept 16, 2026): record 30 submitters, 486 results
- End-to-End RAG: 107,484 passages from 2,515 HTML files, 824 FRAMES multi-hop questions,
  up to 5 retrieval rounds; metrics docs/s (FAISS HNSW build) + tasks/s (QA); Llama-3.1-8B judge, 97% accuracy target
- Edge Agentic: Qwen3.6-27B thinking-off, Q4_K_M GGUF, llama.cpp, 32K context;
  mean latency/turn + TTFT/TPOT; accuracy gated by BFCL v4 at 97% of reference
